<img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:100px" alt="MCUNet Logo">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Déploiement TinyML avec MCUNet </h1></center>
<center><h2> Notebook 03 : Pourquoi pas MobileNet ? (Solution) </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

>
> **Objectif :** Comprendre concrètement pourquoi un modèle "mobile" classique comme MobileNetV2 ne convient pas à un MCU, en analysant la répartition de la mémoire couche par couche.
>
> **Durée estimée :** 45 minutes
>
> **Prérequis :** Notions d'architecture CNN (Convolutions, Inverted Residuals).
>


### Concepts clés
>
> 1. **MobileNetV2** : Architecture de référence pour les téléphones portables (Edge AI). Très efficace en calcul (MACs) mais pas optimisée pour les pics de SRAM.
> 2. **Peak Memory (Pic d'activations)** : Pendant l'exécution d'un réseau couche par couche, la SRAM doit stocker l'entrée de la couche et sa sortie. La taille combinée maximale de ces tenseurs détermine le "Pic SRAM".
> 3. **THOP** : Outil PyTorch pour calculer le nombre de MACs et de paramètres d'un modèle.


--- 
### 1. Chargement des modèles pour comparaison

Nous allons comparer deux modèles conçus pour l'ImageNet, de taille (poids) très similaire : `MobileNetV2-w0.35` et `MCUNet-in2`.

In [ ]:
import torch
import matplotlib.pyplot as plt
from thop import profile
import copy

from mcunet.model_zoo import build_model

# Chargement de MCUNet-in2 (ImageNet)
mcunet_in2, _, _ = build_model(net_id="mcunet-in2", pretrained=False)

# Le Model Zoo MCUNet contient aussi un baseline MobileNetV2 (w0.35, res144)
mbv2, _, _ = build_model(net_id="mbv2-w0.35", pretrained=False)

print("Modèles chargés en mémoire.")

--- 
### 2. Comparaison des métriques statiques (Poids et Calcul)

Utilisons `thop` pour compter les MACs et les paramètres de chaque modèle pour une image d'entrée de 144x144.

In [ ]:
dummy_input_mcunet = torch.randn(1, 3, 160, 160) # mcunet-in2 utilise 160x160
dummy_input_mbv2 = torch.randn(1, 3, 144, 144)   # mbv2-w0.35 utilise 144x144

# Profilage avec THOP
macs_mcunet, params_mcunet = profile(mcunet_in2, inputs=(dummy_input_mcunet,), verbose=False)
macs_mbv2, params_mbv2 = profile(mbv2, inputs=(dummy_input_mbv2,), verbose=False)

print(f"\n--- Comparaison ---")
print(f"MCUNet-in2 : {params_mcunet / 1e6:.2f} M Params | {macs_mcunet / 1e6:.1f} M MACs")
print(f"MobileNetV2: {params_mbv2 / 1e6:.2f} M Params | {macs_mbv2 / 1e6:.1f} M MACs")

Etonnant ! MCUNet a *presque 3 fois plus de MACs* que MobileNetV2, pour un nombre de paramètres similaire. Pourquoi les auteurs de MCUNet affirment-ils qu'il est meilleur pour les MCU ?

C'est parce que le problème principal n'est pas le calcul (ça prend juste plus de temps), c'est la **SRAM** (ça crashe si on dépasse).

--- 
### 3. Profilage de la mémoire d'activation (SRAM)

Nous allons utiliser des *forward hooks* PyTorch pour enregistrer la taille des tenseurs (activations) produits par chaque couche du réseau pendant une inférence.

In [ ]:
activation_sizes = []

def hook_fn(module, input, output):
    # Enregistre la taille du tenseur de sortie en KB (en supposant int8 = 1 byte)
    size_bytes = output.numel() * 1 
    activation_sizes.append(size_bytes / 1024)

def profile_activations(model, dummy_input):
    global activation_sizes
    activation_sizes = []
    
    # On attache le hook à toutes les couches Conv2d
    handles = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            handles.append(module.register_forward_hook(hook_fn))
            
    # On fait une passe avant (sans calculer les gradients)
    with torch.no_grad():
        model(dummy_input)
        
    # On nettoie les hooks
    for handle in handles:
        handle.remove()
        
    return copy.deepcopy(activation_sizes)

sizes_mcunet = profile_activations(mcunet_in2, dummy_input_mcunet)
sizes_mbv2 = profile_activations(mbv2, dummy_input_mbv2)

print(f"Profilage terminé : {len(sizes_mcunet)} couches Conv pour MCUNet, {len(sizes_mbv2)} pour MobileNetV2.")

--- 
### 4. Petit exercice pratique

Tracer un graphique comparant les tailles d'activations couche par couche pour les deux modèles. Identifier le "Peak Memory" pour chacun.

In [ ]:
# Solution
peak_mbv2 = max(sizes_mbv2)
peak_mcunet = max(sizes_mcunet)

print(f"Pic SRAM MobileNetV2 : {peak_mbv2:.1f} KB")
print(f"Pic SRAM MCUNet-in2  : {peak_mcunet:.1f} KB")

plt.figure(figsize=(10, 5))
plt.plot(sizes_mbv2, label=f'MobileNetV2-w0.35 (Peak: {peak_mbv2:.1f} KB)', marker='o', markersize=3)
plt.plot(sizes_mcunet, label=f'MCUNet-in2 (Peak: {peak_mcunet:.1f} KB)', marker='x', markersize=3)

plt.title("Taille des activations par couche (SRAM footprint)")
plt.xlabel("Index de la couche de convolution")
plt.ylabel("Taille de l'activation (KB)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


<div class="alert alert-warning">
<i class='fa fa-exclamation-triangle '></i> &emsp; 
  <strong>Note :</strong> ce calcul est une simplification. Dans la réalité, le moteur d'inférence (comme TinyEngine) a besoin de stocker l'entrée <strong>et</strong> la sortie de la couche simultanément (ping-pong buffers), plus quelques variables de travail. Le vrai pic SRAM est donc plus élevé que la simple plus grosse activation (voir table du Model Zoo). Mais la forme de la courbe reste la même.
</div>

--- 
### Questions de réflexion

1. En regardant le graphique, dans quelle partie du réseau se trouve le pic mémoire de MobileNetV2 (début, milieu, fin) ? Pourquoi de manière conceptuelle (taille spatiale vs nombre de canaux) ?

--> Le pic se trouve au tout début. Les premières couches ont une grande résolution spatiale (144x144, 72x72) même avec peu de canaux. Les activations sont donc énormes. Ensuite le réseau réduit la résolution et augmente les canaux, ce qui réduit la taille des activations.

2. MCUNet a plus de calculs (MACs) mais un pic mémoire plus bas. Comment l'algorithme de recherche (TinyNAS) a-t-il modifié l'architecture de MCUNet par rapport à MobileNetV2 pour accomplir cela ? Qu'est-ce que cela signifie pour la consommation énergétique globale ?

--> TinyNAS a "bridé" l'expansion des premières couches (in-verted bottlenecks très étroits ou résolutions réduites plus vite) pour écrêter le pic initial. En compensation, pour garder une bonne précision, il a ajouté plus de profondeur et de largeur dans les couches profondes (où les activations sont petites). Conséquence : on fait beaucoup plus de calculs au total. Cela consomme plus d'énergie par inférence, mais permet de rentrer dans la SRAM. C'est le compromis typique du TinyML.